# Agentic AI-OPS Driven AMS — MVP Demo
## Automated Managed Services with Clustering, Knowledge Generation & Graph-Based Orchestration

**End-to-End Workflow:**
1. **Ticket Clustering** — Group similar incidents using NLP + ML clustering
2. **Cluster Prioritization** — Rank clusters by volume, severity & recurrence
3. **Knowledge Article (KA) Generation** — Auto-generate structured remediation guides
4. **Enterprise Knowledge Graph (EKG)** — Map relationships between issues, causes, agents & systems
5. **Agent & Remediation Enablement** — Structured workflows for autonomous resolution
6. **Plan Generation & Execution** — Orchestrated execution with risk controls

**Tech Stack:** Python, scikit-learn, NetworkX, Plotly, Matplotlib, Pandas

---

## 0. Setup & Dependencies

In [ ]:
# Install required packages (run once — all standard, no network-dependent models)
import subprocess, sys

packages = [
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn',
    'networkx', 'plotly', 'tabulate'
]

for pkg in packages:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All dependencies installed successfully!')

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import hashlib
from datetime import datetime, timedelta
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix

# Visualization
from IPython.display import display, HTML, Markdown

print('All imports loaded successfully!')
print(f'Demo Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

---
## 1. Data Ingestion — Load Incident Tickets

Loading incident tickets from CSV. In production, this would connect to ServiceNow, SAP Solution Manager, Oracle Cloud, or any ITSM tool via API.

In [ ]:
# Load incident data from CSV
import os

csv_path = 'dummy_incidents.csv'

# Check a few common locations
possible_paths = [
    'dummy_incidents.csv',
    './dummy_incidents.csv',
    os.path.join(os.path.dirname(os.path.abspath('__file__')), 'dummy_incidents.csv')
]

df = None
for p in possible_paths:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f'Loaded data from: {p}')
        break

if df is None:
    raise FileNotFoundError(
        'Could not find dummy_incidents.csv. '
        'Please place it in the same directory as this notebook.'
    )

df['created_date'] = pd.to_datetime(df['created_date'])

print(f'\nLoaded {len(df)} incident tickets')
print(f'Date range: {df["created_date"].min().date()} to {df["created_date"].max().date()}')
print(f'Systems: {df["system"].unique().tolist()}')
print(f'Modules: {df["module"].nunique()} unique modules')
print(f'\nSeverity distribution:')
print(df['severity'].value_counts().to_string())
display(df.head(10))

In [ ]:
# Generate synthetic OpenTelemetry-style signals/logs linked to tickets
np.random.seed(42)

otel_signals = []
signal_types = ['metric.cpu_usage', 'metric.memory_usage', 'metric.response_time_ms',
                'metric.error_rate', 'log.error', 'log.warning', 'trace.span_duration_ms']

for _, row in df.iterrows():
    n_signals = np.random.randint(3, 8)
    base_time = row['created_date'] - timedelta(hours=np.random.randint(1, 6))
    for i in range(n_signals):
        sig_type = np.random.choice(signal_types)
        if 'cpu' in sig_type:
            value = np.random.uniform(75, 99)
        elif 'memory' in sig_type:
            value = np.random.uniform(80, 98)
        elif 'response_time' in sig_type:
            value = np.random.uniform(2000, 60000)
        elif 'error_rate' in sig_type:
            value = np.random.uniform(5, 50)
        elif 'span_duration' in sig_type:
            value = np.random.uniform(5000, 120000)
        else:
            value = 1.0

        otel_signals.append({
            'ticket_id': row['ticket_id'],
            'timestamp': base_time + timedelta(minutes=i * 5),
            'signal_type': sig_type,
            'value': round(value, 2),
            'service': f"{row['system']}.{row['module']}",
            'environment': row['environment']
        })

df_signals = pd.DataFrame(otel_signals)
print(f'Generated {len(df_signals)} OpenTelemetry signals linked to {df["ticket_id"].nunique()} tickets')
display(df_signals.head(10))

---
## 2. Ticket Clustering — Identify Recurring Patterns

We combine text features (descriptions, root causes) with categorical features (system, module, severity) to create rich embeddings, then cluster with KMeans.

In [ ]:
# Build combined text feature for each ticket
df['combined_text'] = (
    df['short_description'] + ' | ' +
    df['detailed_description'] + ' | ' +
    df['root_cause'] + ' | ' +
    df['issue_type'] + ' | ' +
    df['system'] + ' ' + df['module']
)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=200,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=1
)
tfidf_matrix = tfidf.fit_transform(df['combined_text'])

# Add categorical features
le_system = LabelEncoder()
le_module = LabelEncoder()
le_severity = LabelEncoder()
le_issue = LabelEncoder()

cat_features = pd.DataFrame({
    'system_enc': le_system.fit_transform(df['system']),
    'module_enc': le_module.fit_transform(df['module']),
    'severity_enc': le_severity.fit_transform(df['severity']),
    'issue_enc': le_issue.fit_transform(df['issue_type']),
    'recurrence': df['recurrence_count']
})

scaler = StandardScaler()
cat_scaled = scaler.fit_transform(cat_features)

# Combine TF-IDF + categorical features
combined_features = hstack([tfidf_matrix, csr_matrix(cat_scaled)])

print(f'Feature matrix shape: {combined_features.shape}')
print(f'  TF-IDF features: {tfidf_matrix.shape[1]}')
print(f'  Categorical features: {cat_scaled.shape[1]}')

In [ ]:
# Determine optimal number of clusters using silhouette score
silhouette_scores = []
K_range = range(3, 12)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(combined_features.toarray())
    score = silhouette_score(combined_features.toarray(), labels)
    silhouette_scores.append(score)

optimal_k = list(K_range)[np.argmax(silhouette_scores)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(list(K_range), silhouette_scores, 'bo-', linewidth=2, markersize=8)
ax.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal K={optimal_k}')
ax.set_xlabel('Number of Clusters (K)', fontsize=12)
ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Cluster Optimization — Silhouette Analysis', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'\nOptimal number of clusters: {optimal_k} (silhouette score: {max(silhouette_scores):.3f})')

In [ ]:
# Apply KMeans clustering with optimal K
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster_id'] = kmeans.fit_predict(combined_features.toarray())

# PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(combined_features.toarray())
df['pca_x'] = coords_2d[:, 0]
df['pca_y'] = coords_2d[:, 1]

# Name each cluster by its dominant characteristics
cluster_names = {}
for cid in sorted(df['cluster_id'].unique()):
    subset = df[df['cluster_id'] == cid]
    top_system = subset['system'].mode().iloc[0]
    top_module = subset['module'].mode().iloc[0] if len(subset['module'].mode()) > 0 else 'Mixed'
    top_issue = subset['issue_type'].mode().iloc[0]
    cluster_names[cid] = f'C{cid}: {top_system}-{top_module} ({top_issue})'

df['cluster_name'] = df['cluster_id'].map(cluster_names)

# Interactive scatter plot
fig = px.scatter(
    df, x='pca_x', y='pca_y',
    color='cluster_name',
    hover_data=['ticket_id', 'short_description', 'severity', 'system', 'module'],
    title='Incident Ticket Clusters (PCA Projection)',
    labels={'pca_x': 'Principal Component 1', 'pca_y': 'Principal Component 2'},
    width=900, height=600
)
fig.update_traces(marker=dict(size=10, line=dict(width=1, color='white')))
fig.update_layout(legend_title='Cluster')
fig.show()

print('\nCluster Summary:')
for cid, name in sorted(cluster_names.items()):
    count = len(df[df['cluster_id'] == cid])
    print(f'  {name}: {count} tickets')

---
## 3. Cluster Prioritization — Focus Automation Efforts

Rank clusters by a composite score combining ticket volume, severity impact, recurrence frequency, and resolution time.

In [ ]:
# Build prioritization scoring model
severity_weight = {'P1': 4, 'P2': 2.5, 'P3': 1}
impact_weight = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1}

cluster_stats = []
for cid in sorted(df['cluster_id'].unique()):
    subset = df[df['cluster_id'] == cid]
    stats = {
        'cluster_id': cid,
        'cluster_name': cluster_names[cid],
        'ticket_count': len(subset),
        'avg_severity_score': subset['severity'].map(severity_weight).mean(),
        'total_recurrence': subset['recurrence_count'].sum(),
        'avg_recurrence': subset['recurrence_count'].mean(),
        'avg_resolution_hours': subset['resolution_time_hours'].mean(),
        'total_resolution_hours': subset['resolution_time_hours'].sum(),
        'p1_count': len(subset[subset['severity'] == 'P1']),
        'critical_impact_count': len(subset[subset['business_impact'] == 'Critical']),
        'top_systems': ', '.join(subset['system'].value_counts().head(2).index.tolist()),
        'top_modules': ', '.join(subset['module'].value_counts().head(3).index.tolist()),
    }
    cluster_stats.append(stats)

df_clusters = pd.DataFrame(cluster_stats)

# Composite Priority Score (normalized 0-100)
score_cols = ['ticket_count', 'avg_severity_score', 'total_recurrence', 'avg_resolution_hours', 'p1_count']
scaler_priority = MinMaxScaler(feature_range=(0, 100))
normalized = scaler_priority.fit_transform(df_clusters[score_cols])
weights = [0.25, 0.25, 0.20, 0.15, 0.15]
df_clusters['priority_score'] = np.dot(normalized, weights)
df_clusters = df_clusters.sort_values('priority_score', ascending=False).reset_index(drop=True)
df_clusters['priority_rank'] = range(1, len(df_clusters) + 1)

display(HTML('<h3>Cluster Priority Ranking</h3>'))
display(df_clusters[['priority_rank', 'cluster_name', 'ticket_count', 'avg_severity_score',
                      'total_recurrence', 'avg_resolution_hours', 'p1_count', 'priority_score'
                     ]].style.background_gradient(subset=['priority_score'], cmap='YlOrRd')
                      .format({'avg_severity_score': '{:.1f}', 'avg_resolution_hours': '{:.1f}',
                               'priority_score': '{:.1f}', 'avg_recurrence': '{:.1f}'}))

In [ ]:
# Priority visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Cluster Priority Scores', 'Ticket Volume vs Avg Resolution Time'),
    specs=[[{'type': 'bar'}, {'type': 'scatter'}]]
)

colors = px.colors.qualitative.Set2
fig.add_trace(
    go.Bar(
        x=df_clusters['cluster_name'],
        y=df_clusters['priority_score'],
        marker_color=[colors[i % len(colors)] for i in range(len(df_clusters))],
        text=df_clusters['priority_score'].round(1),
        textposition='outside',
        name='Priority Score'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_clusters['ticket_count'],
        y=df_clusters['avg_resolution_hours'],
        mode='markers+text',
        marker=dict(
            size=df_clusters['total_recurrence'] * 1.5,
            color=df_clusters['priority_score'],
            colorscale='YlOrRd',
            showscale=True,
            colorbar=dict(title='Priority')
        ),
        text=[f'C{i}' for i in df_clusters['cluster_id']],
        textposition='top center',
        name='Clusters'
    ),
    row=1, col=2
)

fig.update_layout(height=450, width=1100, showlegend=False,
                  title_text='Cluster Prioritization Dashboard')
fig.update_xaxes(title_text='Cluster', row=1, col=1, tickangle=-30)
fig.update_yaxes(title_text='Priority Score', row=1, col=1)
fig.update_xaxes(title_text='Ticket Count', row=1, col=2)
fig.update_yaxes(title_text='Avg Resolution Time (hrs)', row=1, col=2)
fig.show()

---
## 4. Knowledge Article (KA) Generation

Automatically generate structured Knowledge Articles for each prioritized cluster. KAs serve as algorithmic guidance for the AI-OPS Brain and step-by-step remediation instructions for agents.

In [ ]:
# Knowledge Article Generator
class KnowledgeArticleGenerator:
    """Generates structured Knowledge Articles from incident clusters."""

    def __init__(self, df, cluster_stats):
        self.df = df
        self.cluster_stats = cluster_stats
        self.ka_store = {}  # version-controlled KA storage

    def generate_ka(self, cluster_id):
        """Generate a Knowledge Article for a given cluster."""
        subset = self.df[self.df['cluster_id'] == cluster_id]
        stats = self.cluster_stats[self.cluster_stats['cluster_id'] == cluster_id].iloc[0]

        # Extract common root causes
        root_causes = subset['root_cause'].value_counts()

        # Extract common remediation patterns
        all_steps = []
        for steps in subset['remediation_steps']:
            for step in steps.split('. '):
                step = step.strip()
                if step and len(step) > 5:
                    cleaned = step.lstrip('0123456789').lstrip('. ')
                    if cleaned:
                        all_steps.append(cleaned)

        step_freq = Counter(all_steps)
        top_steps = step_freq.most_common(8)

        # Identify agents
        agents = subset['assigned_agent'].value_counts()

        ka = {
            'ka_id': f'KA-{cluster_id:03d}',
            'version': '1.0',
            'created_date': datetime.now().isoformat(),
            'cluster_id': cluster_id,
            'cluster_name': stats['cluster_name'],
            'title': f"Remediation Guide: {stats['cluster_name']}",
            'priority_score': float(stats['priority_score']),
            'ticket_count': int(stats['ticket_count']),
            'total_recurrence': int(stats['total_recurrence']),

            'rca_summary': {
                'primary_causes': {k: int(v) for k, v in root_causes.head(3).items()},
                'affected_systems': stats['top_systems'],
                'affected_modules': stats['top_modules']
            },

            'remediation_steps': [
                {'step': i+1, 'action': step, 'frequency': count}
                for i, (step, count) in enumerate(top_steps)
            ],

            'suggested_agents': [
                {'agent': agent, 'ticket_count': int(cnt)}
                for agent, cnt in agents.items()
            ],

            'validation_steps': [
                'Verify system logs show no recurring errors for 24 hours post-fix',
                'Confirm affected transactions complete within SLA thresholds',
                'Run regression test on related business processes',
                'Monitor OpenTelemetry signals for anomaly recurrence'
            ],

            'automation_potential': (
                'High' if stats['total_recurrence'] > 30 else
                'Medium' if stats['total_recurrence'] > 15 else 'Low'
            ),

            'estimated_mttr_reduction': f"{min(60, int(stats['total_recurrence']) * 2)}%"
        }

        self.ka_store[ka['ka_id']] = ka
        return ka

    def generate_all(self):
        """Generate KAs for all clusters."""
        for cid in sorted(self.df['cluster_id'].unique()):
            self.generate_ka(cid)
        return self.ka_store

    def display_ka(self, ka_id):
        """Pretty-print a Knowledge Article."""
        ka = self.ka_store[ka_id]
        md = f"""### {ka['title']}
**KA ID:** {ka['ka_id']} | **Version:** {ka['version']} | **Priority Score:** {ka['priority_score']:.1f}  
**Tickets in Cluster:** {ka['ticket_count']} | **Total Recurrence:** {ka['total_recurrence']} | **Automation Potential:** {ka['automation_potential']}

#### Root Cause Analysis
**Affected Systems:** {ka['rca_summary']['affected_systems']}  
**Affected Modules:** {ka['rca_summary']['affected_modules']}

**Primary Root Causes:**
"""
        for cause, count in ka['rca_summary']['primary_causes'].items():
            md += f"- ({count}x) {cause}\n"

        md += "\n#### Recommended Remediation Steps\n"
        for step in ka['remediation_steps']:
            md += f"{step['step']}. {step['action']}\n"

        md += "\n#### Suggested Agents\n"
        for agent in ka['suggested_agents']:
            md += f"- **{agent['agent']}** ({agent['ticket_count']} tickets)\n"

        md += "\n#### Validation Steps\n"
        for i, vs in enumerate(ka['validation_steps'], 1):
            md += f"{i}. {vs}\n"

        md += f"\n**Estimated MTTR Reduction:** {ka['estimated_mttr_reduction']}\n"
        display(Markdown(md))


# Generate all Knowledge Articles
ka_gen = KnowledgeArticleGenerator(df, df_clusters)
all_kas = ka_gen.generate_all()

print(f'Generated {len(all_kas)} Knowledge Articles')
print(f'KA IDs: {list(all_kas.keys())}')

In [ ]:
# Display all Knowledge Articles (sorted by priority)
sorted_clusters = df_clusters.sort_values('priority_score', ascending=False)

for _, row in sorted_clusters.iterrows():
    ka_id = f"KA-{int(row['cluster_id']):03d}"
    ka_gen.display_ka(ka_id)
    display(HTML('<hr style="border: 2px solid #ccc;">'))

---
## 5. Enterprise Knowledge Graph (EKG) Generation

Build a graph mapping relationships between Issues, Root Causes, Remediation Steps, Agents, Systems, and Signals.

In [ ]:
# Build Enterprise Knowledge Graph
G = nx.DiGraph()

# Color scheme for node types
node_colors_map = {
    'system': '#FF6B6B',
    'module': '#4ECDC4',
    'issue_type': '#45B7D1',
    'root_cause': '#FFA07A',
    'agent': '#98D8C8',
    'cluster': '#DDA0DD',
    'ka': '#FFD700',
    'signal': '#87CEEB'
}

# Add nodes and edges from incident data
for _, row in df.iterrows():
    # System node
    sys_id = f"SYS:{row['system']}"
    G.add_node(sys_id, type='system', label=row['system'],
               color=node_colors_map['system'])

    # Module node
    mod_id = f"MOD:{row['system']}.{row['module']}"
    G.add_node(mod_id, type='module', label=f"{row['system']}.{row['module']}",
               color=node_colors_map['module'])
    G.add_edge(sys_id, mod_id, relation='has_module')

    # Issue type node
    issue_id = f"ISS:{row['issue_type']}"
    G.add_node(issue_id, type='issue_type', label=row['issue_type'],
               color=node_colors_map['issue_type'])
    G.add_edge(mod_id, issue_id, relation='experiences_issue',
               weight=row['recurrence_count'])

    # Root cause node
    rc_short = row['root_cause'][:60]
    rc_id = f"RC:{hashlib.md5(row['root_cause'].encode()).hexdigest()[:8]}"
    G.add_node(rc_id, type='root_cause', label=rc_short,
               full_text=row['root_cause'], color=node_colors_map['root_cause'])
    G.add_edge(issue_id, rc_id, relation='caused_by')

    # Agent node
    agent_id = f"AGT:{row['assigned_agent']}"
    G.add_node(agent_id, type='agent', label=row['assigned_agent'],
               color=node_colors_map['agent'])
    G.add_edge(rc_id, agent_id, relation='resolved_by')

    # Cluster node
    cl_id = f"CL:{row['cluster_id']}"
    G.add_node(cl_id, type='cluster', label=cluster_names[row['cluster_id']],
               color=node_colors_map['cluster'])
    G.add_edge(rc_id, cl_id, relation='belongs_to_cluster')

# Add KA nodes linked to clusters
for ka_id, ka in all_kas.items():
    ka_node = f"KA:{ka_id}"
    G.add_node(ka_node, type='ka', label=ka_id,
               color=node_colors_map['ka'], priority=ka['priority_score'])
    cl_id = f"CL:{ka['cluster_id']}"
    G.add_edge(cl_id, ka_node, relation='documented_in')

# Add signal type nodes from OTel data
for sig_type in df_signals['signal_type'].unique():
    sig_id = f"SIG:{sig_type}"
    G.add_node(sig_id, type='signal', label=sig_type,
               color=node_colors_map['signal'])

# Connect signals to modules
for service in df_signals['service'].unique():
    mod_id = f"MOD:{service}"
    if mod_id in G.nodes:
        for sig_type in df_signals[df_signals['service'] == service]['signal_type'].unique():
            sig_id = f"SIG:{sig_type}"
            G.add_edge(mod_id, sig_id, relation='emits_signal')

print(f'Enterprise Knowledge Graph built!')
print(f'  Nodes: {G.number_of_nodes()}')
print(f'  Edges: {G.number_of_edges()}')
print(f'  Node types: {dict(Counter(nx.get_node_attributes(G, "type").values()))}')

In [ ]:
# Visualize the EKG
fig, ax = plt.subplots(figsize=(18, 14))

pos = nx.spring_layout(G, k=2.5, iterations=50, seed=42)

# Draw edges
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.15, edge_color='gray',
                        arrows=True, arrowsize=8, width=0.5)

# Draw nodes by type
node_type_map = nx.get_node_attributes(G, 'type')
size_map = {'system': 600, 'module': 400, 'issue_type': 350, 'root_cause': 200,
            'agent': 350, 'cluster': 500, 'ka': 450, 'signal': 250}

for ntype, color in node_colors_map.items():
    nodes = [n for n, t in node_type_map.items() if t == ntype]
    if nodes:
        nx.draw_networkx_nodes(G, pos, nodelist=nodes, ax=ax,
                               node_color=color, node_size=size_map.get(ntype, 200),
                               alpha=0.85, edgecolors='white', linewidths=1)

# Labels for important nodes
important_types = {'system', 'module', 'agent', 'ka', 'cluster', 'issue_type', 'signal'}
labels = {n: d.get('label', n)[:25] for n, d in G.nodes(data=True)
          if d.get('type') in important_types}
nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=6, font_weight='bold')

# Legend
legend_patches = [mpatches.Patch(color=c, label=t.replace('_', ' ').title())
                  for t, c in node_colors_map.items()]
ax.legend(handles=legend_patches, loc='upper left', fontsize=9, framealpha=0.9)

ax.set_title('Enterprise Knowledge Graph (EKG)\nIssues > Root Causes > Agents > Clusters > Knowledge Articles',
             fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# EKG Analytics
print('=== EKG Analytics ===')
print(f'\nGraph Density: {nx.density(G):.4f}')

# Most connected nodes (hubs)
degree_centrality = nx.degree_centrality(G)
top_hubs = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
print('\nTop 10 Hub Nodes (highest connectivity):')
for node, centrality in top_hubs:
    node_data = G.nodes[node]
    print(f"  {node_data.get('label', node)[:40]:40s} | Type: {node_data.get('type', '?'):12s} | Centrality: {centrality:.3f}")

# Critical path examples
print('\nSample Paths — System to Knowledge Article:')
system_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'system']
ka_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'ka']

paths_shown = 0
for sn in system_nodes:
    for kn in ka_nodes:
        try:
            path = nx.shortest_path(G, sn, kn)
            path_labels = [G.nodes[n].get('label', n)[:30] for n in path]
            print(f'  {" -> ".join(path_labels)}')
            paths_shown += 1
            if paths_shown >= 3:
                break
        except nx.NetworkXNoPath:
            continue
    if paths_shown >= 3:
        break

---
## 6. Vector Store — Semantic Search for KAs

Using TF-IDF cosine similarity as a lightweight vector store. When a new incident arrives, we find the most relevant KA instantly. (In production, swap with ChromaDB, Pinecone, or Weaviate.)

In [ ]:
# Build TF-IDF Vector Store for Knowledge Articles
class KAVectorStore:
    """Lightweight vector store using TF-IDF + cosine similarity.
    Drop-in replacement for ChromaDB in MVP. Zero network dependencies."""

    def __init__(self):
        self.documents = []
        self.metadatas = []
        self.ids = []
        self.vectorizer = TfidfVectorizer(
            max_features=500, stop_words='english',
            ngram_range=(1, 2), min_df=1
        )
        self.tfidf_matrix = None

    def add(self, documents, metadatas, ids):
        self.documents.extend(documents)
        self.metadatas.extend(metadatas)
        self.ids.extend(ids)

    def build_index(self):
        self.tfidf_matrix = self.vectorizer.fit_transform(self.documents)
        print(f'Vector index built: {self.tfidf_matrix.shape[0]} documents, '
              f'{self.tfidf_matrix.shape[1]} features')

    def query(self, query_texts, n_results=2):
        query_vec = self.vectorizer.transform(query_texts)
        similarities = cosine_similarity(query_vec, self.tfidf_matrix)[0]

        top_indices = similarities.argsort()[::-1][:n_results]

        return {
            'documents': [[self.documents[i] for i in top_indices]],
            'metadatas': [[self.metadatas[i] for i in top_indices]],
            'distances': [[round(1 - similarities[i], 4) for i in top_indices]],
            'ids': [[self.ids[i] for i in top_indices]]
        }

    def count(self):
        return len(self.documents)


# Initialize and populate the vector store
ka_collection = KAVectorStore()

for ka_id, ka in all_kas.items():
    ka_text = (f"{ka['title']}. "
               f"Systems: {ka['rca_summary']['affected_systems']}. "
               f"Modules: {ka['rca_summary']['affected_modules']}. "
               f"Root causes: {', '.join(ka['rca_summary']['primary_causes'].keys())}. "
               f"Steps: {', '.join(s['action'] for s in ka['remediation_steps'][:5])}. "
               f"Automation potential: {ka['automation_potential']}.")

    ka_collection.add(
        documents=[ka_text],
        metadatas=[{
            'ka_id': ka_id,
            'cluster_id': str(ka['cluster_id']),
            'priority_score': str(round(ka['priority_score'], 1)),
            'automation_potential': ka['automation_potential'],
            'ticket_count': str(ka['ticket_count'])
        }],
        ids=[ka_id]
    )

ka_collection.build_index()
print(f'Loaded {ka_collection.count()} Knowledge Articles into vector store')

In [ ]:
# Semantic Search Demo — Find relevant KA for new incidents
test_incidents = [
    "SAP system performance degraded, all transactions slow, database response times high",
    "Oracle AP invoice import failing with duplicate error, invoices stuck in interface",
    "Users cannot post financial documents, period not open error in SAP FI",
    "Warehouse RF scanners timing out during goods receipt, WM transactions frozen",
    "Background job queue full, 300+ jobs in Released status not executing"
]

display(HTML('<h3>Semantic KA Search — New Incident Matching</h3>'))

for incident_desc in test_incidents:
    results = ka_collection.query(
        query_texts=[incident_desc],
        n_results=2
    )

    print(f'\nNew Incident: "{incident_desc[:70]}..."')
    for i, (doc, meta, dist) in enumerate(
        zip(results['documents'][0], results['metadatas'][0], results['distances'][0])
    ):
        confidence = max(0, (1 - dist)) * 100
        print(f'   Match {i+1}: {meta["ka_id"]} (confidence: {confidence:.0f}%) | '
              f'Automation: {meta["automation_potential"]} | Tickets: {meta["ticket_count"]}')
    print('   ---')

---
## 7. Agent & Remediation Enablement

Define structured agent workflows that use KAs as step-by-step guides. Each agent has capabilities, assigned clusters, and autonomy levels.

In [ ]:
# AI-OPS Agent Registry
class AgentRegistry:
    """Registry of all automation agents with their capabilities."""

    def __init__(self):
        self.agents = {}
        self._build_agents()

    def _build_agents(self):
        agent_defs = [
            {
                'agent_id': 'Agent_DB_01',
                'name': 'Database Performance Agent',
                'type': 'AI',
                'capabilities': ['index_optimization', 'buffer_tuning', 'query_analysis',
                                 'table_statistics', 'lock_resolution', 'memory_management'],
                'autonomy_level': 'Semi-Autonomous',
                'risk_tolerance': 'Medium',
                'systems': ['SAP ECC', 'Oracle EBS', 'SAP HANA']
            },
            {
                'agent_id': 'Agent_FI_01',
                'name': 'Financial Operations Agent',
                'type': 'AI',
                'capabilities': ['period_management', 'payment_processing', 'clearing_automation',
                                 'tax_configuration', 'bank_reconciliation'],
                'autonomy_level': 'Human-Supervised',
                'risk_tolerance': 'Low',
                'systems': ['SAP ECC', 'Oracle EBS']
            },
            {
                'agent_id': 'Agent_SD_01',
                'name': 'Sales & Distribution Agent',
                'type': 'AI',
                'capabilities': ['pricing_config', 'output_determination', 'partner_management',
                                 'credit_management', 'billing_processing'],
                'autonomy_level': 'Semi-Autonomous',
                'risk_tolerance': 'Medium',
                'systems': ['SAP ECC']
            },
            {
                'agent_id': 'Agent_BASIS_01',
                'name': 'Basis & Infrastructure Agent',
                'type': 'AI',
                'capabilities': ['job_scheduling', 'transport_management', 'spool_management',
                                 'rfc_monitoring', 'system_monitoring', 'memory_management'],
                'autonomy_level': 'Autonomous',
                'risk_tolerance': 'Medium',
                'systems': ['SAP ECC', 'SAP HANA']
            },
            {
                'agent_id': 'Agent_MM_01',
                'name': 'Materials Management Agent',
                'type': 'AI',
                'capabilities': ['po_management', 'inventory_control', 'batch_management',
                                 'vendor_evaluation', 'mrp_processing'],
                'autonomy_level': 'Semi-Autonomous',
                'risk_tolerance': 'Medium',
                'systems': ['SAP ECC', 'Oracle EBS']
            },
            {
                'agent_id': 'Agent_SEC_01',
                'name': 'Security & Access Agent',
                'type': 'AI',
                'capabilities': ['user_management', 'role_assignment', 'auth_troubleshooting',
                                 'password_management'],
                'autonomy_level': 'Human-Supervised',
                'risk_tolerance': 'Low',
                'systems': ['SAP ECC', 'Oracle EBS']
            },
            {
                'agent_id': 'Agent_HR_01',
                'name': 'HR Operations Agent',
                'type': 'AI',
                'capabilities': ['absence_management', 'payroll_support', 'org_management'],
                'autonomy_level': 'Human-Supervised',
                'risk_tolerance': 'Low',
                'systems': ['Oracle EBS']
            }
        ]

        for a in agent_defs:
            self.agents[a['agent_id']] = a

    def get_agent(self, agent_id):
        return self.agents.get(agent_id)

    def find_capable_agents(self, required_capability):
        return [
            a for a in self.agents.values()
            if required_capability.lower() in ' '.join(a['capabilities']).lower()
        ]

    def display_all(self):
        rows = []
        for a in self.agents.values():
            rows.append({
                'Agent ID': a['agent_id'],
                'Name': a['name'],
                'Autonomy': a['autonomy_level'],
                'Risk Tolerance': a['risk_tolerance'],
                'Capabilities': ', '.join(a['capabilities'][:4]) + '...',
                'Systems': ', '.join(a['systems'])
            })
        return pd.DataFrame(rows)


registry = AgentRegistry()
display(HTML('<h3>AI-OPS Agent Registry</h3>'))
display(registry.display_all().style.set_properties(**{'text-align': 'left'}))

In [ ]:
# Build Remediation Workflow from KA
class RemediationWorkflow:
    """Structured remediation workflow driven by Knowledge Articles."""

    def __init__(self, ka, agent_registry):
        self.ka = ka
        self.registry = agent_registry
        self.workflow_steps = []
        self._build_workflow()

    def _build_workflow(self):
        agents = self.ka['suggested_agents']
        primary_agent = agents[0]['agent'] if agents else 'Unassigned'

        for step in self.ka['remediation_steps']:
            action = step['action']
            can_automate = any(
                cap in action.lower()
                for cap in ['check', 'monitor', 'run', 'analyze', 'query', 'verify', 'review']
            )
            needs_approval = any(
                word in action.lower()
                for word in ['update', 'change', 'modify', 'delete', 'restart', 'reverse', 'mass']
            )

            self.workflow_steps.append({
                'step_num': step['step'],
                'action': action,
                'assigned_agent': primary_agent,
                'execution_mode': (
                    'Automated' if can_automate and not needs_approval
                    else 'Requires Approval' if needs_approval
                    else 'Semi-Automated'
                ),
                'risk_level': 'High' if needs_approval else 'Low',
                'estimated_time': f"{np.random.randint(2, 15)} min"
            })

        for i, vs in enumerate(self.ka['validation_steps']):
            self.workflow_steps.append({
                'step_num': len(self.ka['remediation_steps']) + i + 1,
                'action': f'[VALIDATION] {vs}',
                'assigned_agent': primary_agent,
                'execution_mode': 'Automated',
                'risk_level': 'Low',
                'estimated_time': f"{np.random.randint(5, 20)} min"
            })

    def display(self):
        display(HTML(f"<h4>Workflow: {self.ka['title']}</h4>"))
        df_wf = pd.DataFrame(self.workflow_steps)
        display(df_wf.style.apply(
            lambda row: [
                'background-color: #d4edda' if row['execution_mode'] == 'Automated'
                else 'background-color: #fff3cd' if row['execution_mode'] == 'Semi-Automated'
                else 'background-color: #f8d7da'
            ] * len(row), axis=1
        ))
        auto_count = sum(1 for s in self.workflow_steps if s['execution_mode'] == 'Automated')
        total = len(self.workflow_steps)
        print(f'\nAutomation Coverage: {auto_count}/{total} steps ({auto_count/total*100:.0f}%)')


# Display workflows for all clusters
for _, row in df_clusters.iterrows():
    ka_id = f"KA-{int(row['cluster_id']):03d}"
    ka = all_kas[ka_id]
    workflow = RemediationWorkflow(ka, registry)
    workflow.display()
    print()

---
## 8. AI-OPS Brain — Plan Generation & Execution

The AI-OPS Brain (orchestrator) generates execution plans using the EKG, assigns agents, monitors outcomes, and adjusts dynamically.

In [ ]:
# AI-OPS Brain — Central Orchestrator
class AIOPSBrain:
    """Central orchestrator that generates and executes resolution plans."""

    def __init__(self, ekg, ka_store, agent_registry, vector_db):
        self.ekg = ekg
        self.ka_store = ka_store
        self.registry = agent_registry
        self.vector_db = vector_db
        self.execution_log = []

    def analyze_incident(self, incident_description, severity='P2'):
        """Analyze a new incident and generate an execution plan."""

        # Step 1: Find matching KA via vector search
        results = self.vector_db.query(
            query_texts=[incident_description],
            n_results=1
        )

        if not results['metadatas'][0]:
            return {'status': 'NO_MATCH', 'message': 'No matching KA found. Escalate to human.'}

        matched_ka_id = results['metadatas'][0][0]['ka_id']
        match_distance = results['distances'][0][0]
        confidence = max(0, (1 - match_distance)) * 100
        ka = self.ka_store[matched_ka_id]

        # Step 2: Determine execution strategy
        if confidence > 60 and ka['automation_potential'] in ['High', 'Medium']:
            strategy = 'AUTONOMOUS'
        elif confidence > 35:
            strategy = 'SUPERVISED'
        else:
            strategy = 'HUMAN_ESCALATION'

        # Step 3: Build execution plan
        plan = {
            'plan_id': f"PLAN-{datetime.now().strftime('%Y%m%d%H%M%S')}",
            'incident_description': incident_description,
            'severity': severity,
            'matched_ka': matched_ka_id,
            'confidence': round(confidence, 1),
            'strategy': strategy,
            'agents': ka['suggested_agents'],
            'steps': [],
            'risk_controls': [],
            'estimated_resolution_time': None
        }

        total_time = 0
        for step in ka['remediation_steps']:
            step_time = np.random.randint(3, 15)
            total_time += step_time
            plan['steps'].append({
                'step_num': step['step'],
                'action': step['action'],
                'agent': ka['suggested_agents'][0]['agent'] if ka['suggested_agents'] else 'human',
                'execution_mode': strategy,
                'estimated_minutes': step_time,
                'status': 'PENDING'
            })

        plan['estimated_resolution_time'] = f"{total_time} minutes"

        plan['risk_controls'] = [
            {'control': 'Pre-execution snapshot/backup', 'status': 'REQUIRED'},
            {'control': 'Rollback plan verified', 'status': 'REQUIRED'},
            {'control': 'Change approval obtained',
             'status': 'REQUIRED' if severity == 'P1' else 'OPTIONAL'},
            {'control': 'Post-execution validation', 'status': 'REQUIRED'},
            {'control': 'OTel signal monitoring (24hr)', 'status': 'REQUIRED'}
        ]

        return plan

    def execute_plan(self, plan):
        """Simulate plan execution with progress tracking."""
        display(Markdown(f"### Executing Plan: {plan['plan_id']}"))
        display(Markdown(f"**Strategy:** {plan['strategy']} | **KA:** {plan['matched_ka']} | "
                        f"**Confidence:** {plan['confidence']}%"))

        print(f"\n{'='*80}")
        print(f"{'Step':>5} | {'Status':^12} | {'Agent':^15} | {'Time':>6} | Action")
        print(f"{'='*80}")

        results = []
        for step in plan['steps']:
            success = np.random.random() > 0.05
            status = 'SUCCESS' if success else 'RETRY'

            if status == 'RETRY':
                success = np.random.random() > 0.2
                status = 'SUCCESS' if success else 'ESCALATED'

            step['status'] = status
            icon = '[OK]' if status == 'SUCCESS' else '[!!]' if status == 'RETRY' else '[XX]'
            print(f"  {step['step_num']:>3} | {icon} {status:<9} | {step['agent']:^15} | "
                  f"{step['estimated_minutes']:>3} min | {step['action'][:50]}")
            results.append(status)

        success_count = results.count('SUCCESS')
        total = len(results)
        print(f"\n{'='*80}")
        print(f"Execution Complete: {success_count}/{total} steps successful")
        print(f"Estimated Resolution Time: {plan['estimated_resolution_time']}")

        print(f"\nRisk Controls:")
        for rc in plan['risk_controls']:
            check = '[x]' if rc['status'] == 'REQUIRED' else '[ ]'
            print(f"  {check} {rc['control']} [{rc['status']}]")

        self.execution_log.append({
            'plan_id': plan['plan_id'],
            'timestamp': datetime.now().isoformat(),
            'success_rate': success_count / total * 100,
            'strategy': plan['strategy']
        })

        return success_count == total


# Initialize the AI-OPS Brain
brain = AIOPSBrain(G, all_kas, registry, ka_collection)
print('AI-OPS Brain initialized and ready!')

In [ ]:
# Demo: Process a new simulated incident end-to-end
new_incident = (
    "SAP system extremely slow, dialog response times over 5 seconds, "
    "users complaining about MM and SD transactions timing out. "
    "HANA memory utilization at 95%. Background jobs are queueing up."
)

display(HTML('<h3>NEW INCIDENT RECEIVED</h3>'))
print(f'Description: {new_incident}')
print(f'Severity: P1')
print()

# Generate plan
plan = brain.analyze_incident(new_incident, severity='P1')

display(HTML(f'<h4>Generated Plan: {plan["plan_id"]}</h4>'))
print(f'Matched KA: {plan["matched_ka"]}')
print(f'Confidence: {plan["confidence"]}%')
print(f'Strategy: {plan["strategy"]}')
print(f'Estimated Time: {plan["estimated_resolution_time"]}')
print()

# Execute plan
success = brain.execute_plan(plan)

In [ ]:
# Process a second incident to show versatility
new_incident_2 = (
    "Oracle AP invoice interface rejecting 200+ invoices with unique constraint violation. "
    "Upstream system sending duplicate invoice numbers. AP team blocked on processing."
)

display(HTML('<h3>SECOND INCIDENT RECEIVED</h3>'))
print(f'Description: {new_incident_2}')
print(f'Severity: P2')
print()

plan2 = brain.analyze_incident(new_incident_2, severity='P2')
print(f'Matched KA: {plan2["matched_ka"]} | Confidence: {plan2["confidence"]}% | Strategy: {plan2["strategy"]}')
print()
brain.execute_plan(plan2)

---
## 9. Executive Dashboard — Expected Outcomes

Visualize KPIs: incident reduction, MTTR improvement, autonomous resolution rate, and operational efficiency.

In [ ]:
# Simulated before/after AI-OPS metrics
np.random.seed(42)
months = pd.date_range('2025-07', '2026-03', freq='MS').strftime('%b %Y').tolist()

incident_counts = [145, 152, 138, 149, 155,   # before AI-OPS
                   128, 105, 87, 72]            # after AI-OPS

mttr_hours = [6.2, 5.8, 6.5, 6.1, 5.9,        # before
              4.8, 3.5, 2.8, 2.1]               # after

auto_resolution_pct = [5, 7, 6, 8, 10,         # before
                       22, 38, 52, 65]           # after

agent_utilization = [95, 93, 96, 94, 92,        # before
                     85, 72, 65, 58]             # after

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Monthly Incident Volume',
        'Mean Time to Resolution (MTTR)',
        'Autonomous Resolution Rate (%)',
        'Human Agent Utilization (%)'
    )
)

colors = ['#3498db'] * 5 + ['#2ecc71'] * 4

fig.add_trace(go.Bar(x=months, y=incident_counts, marker_color=colors,
                     name='Incidents'), row=1, col=1)

fig.add_trace(go.Scatter(x=months, y=mttr_hours, mode='lines+markers',
                         line=dict(color='#e74c3c', width=3),
                         marker=dict(size=10), name='MTTR'), row=1, col=2)

fig.add_trace(go.Bar(x=months, y=auto_resolution_pct, marker_color=colors,
                     name='Auto-Resolve %'), row=2, col=1)

fig.add_trace(go.Scatter(x=months, y=agent_utilization, mode='lines+markers',
                         fill='tozeroy', fillcolor='rgba(52,152,219,0.1)',
                         line=dict(color='#9b59b6', width=3),
                         name='Agent Utilization'), row=2, col=2)

# AI-OPS launch line
for row in [1, 2]:
    for col in [1, 2]:
        fig.add_vline(x='Dec 2025', line_dash='dash', line_color='red',
                      annotation_text='AI-OPS Launch' if (row == 1 and col == 1) else '',
                      row=row, col=col)

fig.update_layout(
    height=600, width=1100,
    title_text='AI-OPS AMS — Executive Dashboard',
    showlegend=False,
    title_font_size=16
)
fig.show()

# Summary KPIs
print('\n' + '='*60)
print('       AI-OPS AMS — Key Performance Indicators')
print('='*60)
print(f'  Incident Reduction:        {(155-72)/155*100:.0f}% down  (155 -> 72/month)')
print(f'  MTTR Improvement:          {(5.9-2.1)/5.9*100:.0f}% down  (5.9 -> 2.1 hours)')
print(f'  Autonomous Resolution:     {65-10}pp up  (10% -> 65%)')
print(f'  Agent Utilization Freed:   {92-58}pp down  (92% -> 58%)')
print(f'  Knowledge Articles:        {len(all_kas)}')
print(f'  EKG Nodes/Edges:           {G.number_of_nodes()}/{G.number_of_edges()}')
print('='*60)

In [ ]:
# Automation Readiness by Cluster
auto_data = []
for _, row in df_clusters.iterrows():
    ka_id = f"KA-{int(row['cluster_id']):03d}"
    ka = all_kas[ka_id]
    wf = RemediationWorkflow(ka, registry)
    auto_steps = sum(1 for s in wf.workflow_steps if s['execution_mode'] == 'Automated')
    total_steps = len(wf.workflow_steps)

    auto_data.append({
        'Cluster': row['cluster_name'],
        'Tickets': int(row['ticket_count']),
        'Recurrence': int(row['total_recurrence']),
        'Automation %': round(auto_steps / total_steps * 100),
        'Priority Score': round(row['priority_score'], 1),
        'Auto Steps': auto_steps,
        'Total Steps': total_steps,
        'Readiness': 'Ready' if auto_steps/total_steps > 0.5 else 'Partial' if auto_steps/total_steps > 0.3 else 'Manual'
    })

df_auto = pd.DataFrame(auto_data)
display(HTML('<h3>Automation Readiness by Cluster</h3>'))
display(df_auto.style.background_gradient(subset=['Automation %', 'Priority Score'], cmap='YlGn')
        .set_properties(**{'text-align': 'center'}))

---
## 10. Architecture Summary

In [ ]:
# Architecture diagram
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')

boxes = [
    (0.5, 7.5, 3.0, 1.5, 'DATA SOURCES\n-----------\nSAP/Oracle Tickets\nOpenTelemetry Signals\nCSV/Excel/APIs', '#E8F5E9'),
    (4.5, 7.5, 3.0, 1.5, 'TICKET CLUSTERING\n-----------\nTF-IDF + KMeans\nPattern Detection\nRecurrence Analysis', '#E3F2FD'),
    (8.5, 7.5, 3.0, 1.5, 'PRIORITIZATION\n-----------\nComposite Scoring\nSeverity x Volume\nImpact Ranking', '#FFF3E0'),
    (12.5, 7.5, 3.0, 1.5, 'KA GENERATION\n-----------\nRCA Summaries\nRemediation Steps\nVersion Control', '#F3E5F5'),
    (2.5, 4.0, 3.5, 1.5, 'ENTERPRISE\nKNOWLEDGE GRAPH\n-----------\nNetworkX Graph DB\nRelationship Mapping', '#FFEBEE'),
    (7.5, 4.0, 3.5, 1.5, 'AI-OPS BRAIN\n(ORCHESTRATOR)\n-----------\nPlan Generation\nDynamic Execution', '#E0F7FA'),
    (12.5, 4.0, 3.0, 1.5, 'AGENT REGISTRY\n-----------\nAI Agents\nHuman Experts\nCapability Matrix', '#F1F8E9'),
    (5.0, 0.5, 6.0, 2.0, 'OUTCOMES\n========================\n53% Less Incidents  |  64% Faster MTTR\n65% Autonomous Resolution\nSelf-Improving AMS Operations', '#FFF9C4'),
]

for (x, y, w, h, label, color) in boxes:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=7.5, fontweight='bold', family='monospace')

# Arrows
arrow_style = dict(arrowstyle='->', color='#555', lw=2)
arrows = [
    ((3.5, 8.25), (4.5, 8.25)),
    ((7.5, 8.25), (8.5, 8.25)),
    ((11.5, 8.25), (12.5, 8.25)),
    ((6.0, 7.5), (4.25, 5.5)),
    ((14.0, 7.5), (14.0, 5.5)),
    ((6.0, 4.75), (7.5, 4.75)),
    ((12.5, 4.75), (11.0, 4.75)),
    ((9.25, 4.0), (8.0, 2.5)),
]

for (start, end) in arrows:
    ax.annotate('', xy=end, xytext=start, arrowprops=arrow_style)

ax.set_title('Agentic AI-OPS AMS — Architecture Overview',
             fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Final summary
display(Markdown("""
## MVP Demo Complete — Summary

| Module | Technology | Status |
|--------|-----------|--------|
| Data Ingestion | Pandas + CSV/Excel | Complete |
| OpenTelemetry Signals | Synthetic OTel data | Complete |
| Ticket Clustering | TF-IDF + KMeans (scikit-learn) | Complete |
| Cluster Prioritization | Composite scoring with MinMaxScaler | Complete |
| Knowledge Article Generation | Automated KA with version control | Complete |
| Enterprise Knowledge Graph | NetworkX (directed graph) | Complete |
| Vector Store / Semantic Search | TF-IDF + Cosine Similarity | Complete |
| Agent Registry & Workflows | Python classes with capability matrix | Complete |
| AI-OPS Brain (Orchestrator) | Plan generation + simulated execution | Complete |
| Executive Dashboard | Plotly interactive charts | Complete |

### Next Steps for Production
1. **Connect real ITSM data** — ServiceNow, SAP SolMan, Jira APIs
2. **Add LLM integration** — Use GPT-4/Claude for richer KA generation and NL plan execution
3. **Persistent storage** — PostgreSQL + Neo4j for EKG, ChromaDB/Pinecone for vectors
4. **Real agent execution** — SSH/API connectors to SAP, Oracle, cloud infrastructure
5. **CI/CD pipeline** — Automated retraining of clusters as new incidents arrive
6. **RBAC & audit trail** — Role-based access and full execution logging
"""))

print('Ready for Monday demo session!')